# Topic discovery experimental notebook

In [23]:
import pandas as pd
import numpy as np
import sqlite3

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.cluster import KMeans

import os
import json
from google import genai


In [24]:
# Get data
conn = sqlite3.connect('../data/processed/dblp.db')

df = pd.read_sql("""
SELECT id, title, year
FROM papers
""", conn)

df = df.dropna(subset=["title"])
df["title"] = df["title"].str.lower()
df = df.sample(100_000, random_state=42) # random sample for faster experiments

In [25]:
# TF-IDF
custom_stopwords = {
    "based", "using", "approach", "method", "model",
    "systems", "system", "study", "analysis", "new",
    "proposed", "results", "paper", "performance"
}
stopwords = list(set(ENGLISH_STOP_WORDS).union(custom_stopwords))

vectorizer = TfidfVectorizer(
    stop_words=stopwords,
    max_features=5000,
    ngram_range=(1, 2),
)

X = vectorizer.fit_transform(df["title"])

In [26]:
# KMeans
k = 8  # number of topics

kmeans = KMeans(n_clusters=k, random_state=42)
df["topic"] = kmeans.fit_predict(X)

In [27]:
# Print top words for each topic
terms = vectorizer.get_feature_names_out()

topics = {}

for i in range(k):
    center = kmeans.cluster_centers_[i]
    top_words = [terms[j] for j in center.argsort()[-10:]]
    topics[i] = top_words
    print(f"Topic {i}: {top_words}")

Topic 0: ['estimation', 'models', 'efficient', 'framework', 'information', 'image', 'time', 'control', 'multi', 'detection']
Topic 1: ['models', 'prediction', 'classification', 'detection', 'reinforcement learning', 'reinforcement', 'deep reinforcement', 'learning', 'deep learning', 'deep']
Topic 2: ['deep neural', 'graph', 'deep', 'convolutional neural', 'convolutional', 'networks', 'neural network', 'neural networks', 'neural', 'network']
Topic 3: ['social', 'control', 'energy', 'social networks', 'wireless sensor', 'sensor networks', 'sensor', 'wireless', 'design', 'networks']
Topic 4: ['detection', 'supervised', 'multi', 'federated learning', 'federated', 'reinforcement', 'reinforcement learning', 'machine', 'machine learning', 'learning']
Topic 5: ['multi objective', 'optimization algorithm', 'swarm', 'improved', 'genetic algorithm', 'objective', 'genetic', 'multi', 'optimization', 'algorithm']
Topic 6: ['multimodal', 'vision', 'vision language', 'large language', 'scale', 'models

In [28]:
# Name topics with Gemini

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def name_topics(topics: dict[int, list[str]]) -> dict[int, str]:
    """
    Send all topics to Gemini in one prompt.
    Returns {topic_id: name}.
    """
    topics_text = "\n".join(
        f"Topic {i}: {', '.join(words)}"
        for i, words in topics.items()
    )

    prompt = f"""You are given {len(topics)} research topics from a computer science paper clustering algorithm.
        Each topic is represented by its 10 most characteristic keywords.

        {topics_text}

        Assign a short, distinct name (2-5 words) to each topic.
        Requirements:
        - Names must be distinct from each other
        - Use standard CS/academic terminology
        - Be specific, not generic (avoid names like "Computer Science Research")
        - Return ONLY a JSON object mapping topic number to name, e.g. {{"0": "Deep Learning Optimization", "1": "Network Security Protocols"}}
    """

    response = client.models.generate_content(
        model="gemini-2.5-pro",
        contents=prompt,
        config={
            'response_mime_type': 'application/json',
        }
    )

    raw = response.text.strip()
    if raw.startswith("```"):
        raw = "\n".join(raw.split("\n")[1:-1])

    names_str = json.loads(raw)
    return {int(k): v for k, v in names_str.items()}

In [29]:
topic_names = name_topics(topics)

for topic_id, name in topic_names.items():
    print(f"Topic {topic_id}: {name}")
    print(f"  Keywords: {topics[topic_id]}")

Topic 0: Efficient Estimation and Detection Models
  Keywords: ['estimation', 'models', 'efficient', 'framework', 'information', 'image', 'time', 'control', 'multi', 'detection']
Topic 1: Deep and Reinforcement Learning Models
  Keywords: ['models', 'prediction', 'classification', 'detection', 'reinforcement learning', 'reinforcement', 'deep reinforcement', 'learning', 'deep learning', 'deep']
Topic 2: Deep Neural Network Architectures
  Keywords: ['deep neural', 'graph', 'deep', 'convolutional neural', 'convolutional', 'networks', 'neural network', 'neural networks', 'neural', 'network']
Topic 3: Wireless Sensor and Social Networks
  Keywords: ['social', 'control', 'energy', 'social networks', 'wireless sensor', 'sensor networks', 'sensor', 'wireless', 'design', 'networks']
Topic 4: Federated and Reinforcement Learning
  Keywords: ['detection', 'supervised', 'multi', 'federated learning', 'federated', 'reinforcement', 'reinforcement learning', 'machine', 'machine learning', 'learning'